In [7]:
# Chunk 2: Data Cleaning & Exploratory Data Analysis (EDA) — Summary
'''
##1. Data Cleaning
- **Removed cancelled/fraud shipments**: Dropped 7,754 rows where `Delivery Status = 'Shipping canceled'`, 
  since these orders were never actually shipped (status: CANCELED or SUSPECTED_FRAUD), and their 
  `Late_delivery_risk = 0` label was misleading (falsely implying "on-time" delivery).
- **Removed PII columns**: Dropped `Customer Email`, `Customer Password`, `Customer Fname`, `Customer Lname`, 
  `Customer Street` — not useful for modeling and a privacy consideration.
- **Removed heavily missing columns**: Dropped `Product Description` (100% missing) and `Order Zipcode` 
  (86% missing).
- **Handled remaining missing values**: Dropped 3 rows with missing `Customer Zipcode`.
- **Fixed data types**: Converted `order date (DateOrders)` and `shipping date (DateOrders)` to proper 
  datetime format.
- **Final cleaned dataset**: 172,762 rows × 46 columns.

## 2. Target Variable Verification
- Verified `Late_delivery_risk` (0/1) against `shipping_gap = Days for shipping (real) - Days for 
  shipment (scheduled)`.
- Found a perfect separation: `shipping_gap ≤ 0` always corresponds to `Late_delivery_risk = 0`, and 
  `shipping_gap ≥ 1` always corresponds to `Late_delivery_risk = 1`.
- **Conclusion**: Target variable is reliable and consistent — safe to use directly for modeling.
- **Data leakage identified**: `Days for shipping (real)` and `shipping_gap` cannot be used as model 
  features, since they are only known after delivery occurs and directly determine the target.

## 3. Class Imbalance Check
- Late (1): 98,976 orders (57.3%)
- Not Late (0): 73,786 orders (42.7%)
- Dataset is reasonably balanced — no special resampling (SMOTE, class weighting) required for modeling.

## 4. Business Question Analysis (EDA Findings)
| Factor | Delay Risk Range | Impact Strength | Key Finding |
|---|---|---|---|
| Shipping Mode | 40% – 100% | 🔴 Very Strong | Tight scheduled delivery windows (e.g., First Class) drive artificially high "late" rates, even when actual delivery speed is comparable to Standard Class |
| Product Price | 40% – 70% | 🟡 Moderate | Non-linear relationship — mid-range priced products (₹806–1204) are riskiest; premium products (₹1600+) are safest |
| Product Category | 58% – 69% | 🟡 Moderate | Niche/specialty categories (Golf Bags & Carts, Lacrosse) show higher delay risk than high-volume categories (Books, Electronics) |
| Order Region | 51% – 59% | 🟢 Weak | Minimal variation across regions — geography is not a strong delay driver |
| Customer State | 59% – 63% | 🟢 Weak | Consistent with region-level finding; sample sizes (159–2,298 orders) verified as sufficiently reliable |
| Customer Segment | 57% – 58% | ⚪ Negligible | No meaningful difference between Consumer, Corporate, and Home Office segments |
| Order Profit Per Order | ₹21.6 vs ₹22.6 | ⚪ Negligible | Late delivery has no meaningful direct impact on recorded profit (profit is determined at order placement, not delivery outcome) |
| Discount Rate | 56% – 58% | ⚪ Negligible | No relationship between discount rate and delay risk |

## 5. Master Insight
Late delivery risk is primarily driven by **operational/logistics factors** (specifically, shipping 
mode scheduling), rather than customer demographics, geography, or pricing. The most actionable lever 
for reducing late delivery is setting more realistic scheduled delivery windows for fast-shipping modes 
(First Class, Second Class), rather than targeting specific regions, customer segments, or price tiers.

## 6. Known Limitations
- Order cancellation patterns (Business Question E) could not be explored, since cancelled/fraud orders 
  were removed during data cleaning to preserve target variable integrity.
- Financial impact analysis only captures direct/recorded profit; indirect costs of late delivery 
  (customer churn, negative reviews, complaint handling) are not present in this dataset.
  '''

'\n##1. Data Cleaning\n- **Removed cancelled/fraud shipments**: Dropped 7,754 rows where `Delivery Status = \'Shipping canceled\'`, \n  since these orders were never actually shipped (status: CANCELED or SUSPECTED_FRAUD), and their \n  `Late_delivery_risk = 0` label was misleading (falsely implying "on-time" delivery).\n- **Removed PII columns**: Dropped `Customer Email`, `Customer Password`, `Customer Fname`, `Customer Lname`, \n  `Customer Street` — not useful for modeling and a privacy consideration.\n- **Removed heavily missing columns**: Dropped `Product Description` (100% missing) and `Order Zipcode` \n  (86% missing).\n- **Handled remaining missing values**: Dropped 3 rows with missing `Customer Zipcode`.\n- **Fixed data types**: Converted `order date (DateOrders)` and `shipping date (DateOrders)` to proper \n  datetime format.\n- **Final cleaned dataset**: 172,762 rows × 46 columns.\n\n## 2. Target Variable Verification\n- Verified `Late_delivery_risk` (0/1) against `shipping_g

In [8]:
# Chunk 3: Feature Engineering — Summary

'''## 3A: Lead Time Variance
- Attempted at Shipping Mode level → constant (0.0), rejected as useless.
- Successfully created `category_shipping_variability`: std deviation of scheduled 
  shipping days per Category Name (range 1.33–1.58), capturing mixed shipping-mode 
  usage per category.

## 3B: Seasonal/Time-Based Features
- Created `order_month`, `order_dayofweek`, `order_quarter`, `order_is_weekend` from 
  `order date (DateOrders)`.
- Business check showed weak/negligible impact on delay risk (weekend vs weekday: 
  57.3% vs 57.2%; months: 56.7%–58.0%), consistent with earlier findings that delay 
  is operationally driven, not time-driven. Retained anyway for potential model 
  interaction effects.

## 3C: Historical Reliability Scores
- Created `category_historical_delay_rate`, `state_historical_delay_rate`, and 
  `region_historical_delay_rate` — average historical delay rate per Category/State/
  Region, giving the model a direct numeric risk signal.
- Note: must be recalculated using only training data after train-test split, to 
  avoid data leakage.

## 3D: Categorical Encoding
- Dropped leakage-risk columns: `Delivery Status` (directly derived from target), 
  `Days for shipping (real)`, `shipping_gap` (removed earlier).
- Verified `Order Status` was safe (55.6%–57.9% delay rate range, no extreme values) 
  before encoding.
- Dropped high-cardinality, low-value columns: `Customer City`, `Order City`, 
  `Order Country`, `Order State`, `Product Image`, `Product Name`.
- One-hot encoded manageable categorical columns: `Type`, `Customer Country`, 
  `Department Name`, `Market`, `Order Status`.
- Replaced high-cardinality columns (`Category Name`, `Customer State`, `Order Region`) 
  with historical reliability score features instead of one-hot encoding.

## Final Dataset
- Shape: 172,762 rows × 65 columns
- Fully numeric — ready for model training.'''

'## 3A: Lead Time Variance\n- Attempted at Shipping Mode level → constant (0.0), rejected as useless.\n- Successfully created `category_shipping_variability`: std deviation of scheduled \n  shipping days per Category Name (range 1.33–1.58), capturing mixed shipping-mode \n  usage per category.\n\n## 3B: Seasonal/Time-Based Features\n- Created `order_month`, `order_dayofweek`, `order_quarter`, `order_is_weekend` from \n  `order date (DateOrders)`.\n- Business check showed weak/negligible impact on delay risk (weekend vs weekday: \n  57.3% vs 57.2%; months: 56.7%–58.0%), consistent with earlier findings that delay \n  is operationally driven, not time-driven. Retained anyway for potential model \n  interaction effects.\n\n## 3C: Historical Reliability Scores\n- Created `category_historical_delay_rate`, `state_historical_delay_rate`, and \n  `region_historical_delay_rate` — average historical delay rate per Category/State/\n  Region, giving the model a direct numeric risk signal.\n- Note: